# 03 — Tracing: build the spans, then meet Phoenix

**What you'll learn**

- What a span is — a named, timed tree node — and why `shoplab.trace.Span` needs exactly seven fields
- How a stack of open spans turns `with tracer.span(...)` nesting into parent pointers, and what `@tracer.traced` adds
- How to trace a real `run_agent` ticket run by wrapping the call site and the tool registry — no internals patched
- What OTLP actually is on the wire, and how `export_phoenix` lands your hand-built spans in the Phoenix UI
- What `obs.enable_phoenix()` has been doing since chapter 00: OpenInference autoinstrumentation, and where it is blind

*Time: ~10 min. Cost: ~$0.002. Cached reruns are free.*

Agents fail in the middle. Chapter 02's loop can make nine model calls, run eight tools, and hand you one wrong decision — and if that final dict is all you kept, the failure is unstudyable: which policy came back from search, what the model did with it, where the thirty seconds went, all gone. A trace is the flight recorder for one run: every operation written down as a named, timed node in a tree, while it happens, readable after the crash.

Observability vendors will happily sell you this as magic. It is not — it is dataclasses with timestamps, a stack, and an HTTP POST. So this chapter applies the course's frameworks-are-packaging lesson to the observability layer itself: build the spans by hand, nest them with a tracer that fits in one cell, ship them to Phoenix with a single POST, and only then turn on the industrial machinery — which, it will turn out, has been running since the cell below.

In [ ]:
# === config (identical in every notebook) ===
import os, getpass
import litellm
from dotenv import load_dotenv              # pip install -e ".[obs]" if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead

TEMPERATURE = 0                             # the whole course runs at temperature 0
litellm.drop_params = True                  # ignore params a provider does not support
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=".litellm_cache")  # reruns are ~free

### Phoenix observability (optional)

Optional in every other chapter; today it is the subject. Run it: this notebook posts spans to the server it starts and ends inside its UI. Note the link it prints — you will open it mid-chapter.

In [ ]:
# optional: Phoenix tracing (see notebook 03)
import obs

obs.enable_phoenix()

## A span is a dict with opinions

One timed operation needs surprisingly little: a `name` (what happened), a `kind` (which sort of actor — this course uses `agent`, `llm`, and `tool`, the vocabulary Phoenix colors its UI by), `start` and `end` wall-clock times, an `attributes` dict for anything worth remembering about the operation, and two ids. The ids are the entire trick: every span carries its own `span_id` plus a `parent_id` naming the span it happened *inside*. That one pointer turns a flat list of spans into a tree.

The cell below is not a sketch — it ships verbatim as `shoplab.trace.Span`, and the build validator byte-compares this cell against the package source, so the two can never drift.

In [ ]:
import time, uuid
from dataclasses import dataclass


# >>> shoplab.trace.Span
@dataclass
class Span:
    """One timed operation: a node in the trace tree."""
    name: str
    kind: str
    start: float
    end: float | None
    attributes: dict
    span_id: str
    parent_id: str | None

    @property
    def duration_ms(self):
        return None if self.end is None else round((self.end - self.start) * 1000, 3)
# <<< shoplab.trace.Span

In [ ]:
agent_span = Span(name="triage", kind="agent", start=time.time(), end=None,
                  attributes={"ticket": "TKT-2205"}, span_id=uuid.uuid4().hex[:16],
                  parent_id=None)
tool_span = Span(name="policy_lookup", kind="tool", start=time.time(), end=None,
                 attributes={}, span_id=uuid.uuid4().hex[:16],
                 parent_id=agent_span.span_id)
time.sleep(0.03)
tool_span.end = time.time()
agent_span.end = time.time()
print(agent_span.name, agent_span.duration_ms, "ms /",
      tool_span.name, tool_span.duration_ms, "ms")
print("ids:", agent_span.span_id, tool_span.span_id,
      "nested:", tool_span.parent_id == agent_span.span_id)

> **What you should see:** both durations at least 30 ms (the sleep), the child no longer than the parent it ran inside, and `nested: True`. Each id is 16 hex characters — unique enough within a trace, and exactly the width OTLP will expect later.

## The tracer is a stack

Writing `parent_id` by hand does not survive contact with real code — by the third nested operation you are threading span objects through every function signature. The fix is one observation: at any moment, the span that should adopt new children is *whichever span is open right now*. That is a stack. Push on enter, pop on exit, and the top of the stack is the next span's parent; a `contextmanager` does the push/pop and stamps both times.

In [ ]:
from contextlib import contextmanager

class MiniTracer:
    def __init__(self):
        self.spans, self._stack = [], []

    @contextmanager
    def span(self, name, kind="span", **attrs):
        s = Span(name, kind, time.time(), None, dict(attrs), uuid.uuid4().hex[:16],
                 self._stack[-1].span_id if self._stack else None)
        self.spans.append(s)
        self._stack.append(s)
        try:
            yield s
        finally:
            s.end = time.time()
            self._stack.pop()

That sketch is the whole idea. The shipped `shoplab.trace.Tracer` keeps this body and adds the conveniences the rest of the chapter leans on: `@tracer.traced` (decorator form — one span per call, named after the function), `print_tree()` (one line per span, indented by depth), and `clear()`. Import the real one and nest three spans, siblings included.

In [ ]:
from shoplab.trace import Tracer

tracer = Tracer()
with tracer.span("triage", "agent", ticket="TKT-2205"):
    with tracer.span("model call", "llm"):
        time.sleep(0.02)
    with tracer.span("policy_lookup", "tool"):
        time.sleep(0.01)
tracer.print_tree()

> **What you should see:** `triage` at the left margin with its `ticket` attribute on the line, and *two* children indented one level — `model call` around 20 ms, `policy_lookup` around 10 ms. The second child sits under `triage`, not under `model call`: the stack popped when the first `with` block closed. That pop is the entire difference between a log and a trace.

## Wrap the call site, not the internals

Now the real thing: chapter 02's `run_agent`, the nine standard tools, and ticket `TKT-2205`. The temptation is to open up the loop and sprinkle spans through it — resist. Two wraps at edges you already own get the whole tree. The agent span goes around the call site: anything recorded while `with tracer.span("triage", "agent")` is open becomes its child, and because the block yields the span, you can attach attributes like the stop reason on the way out.

Tool spans come from the registry. Each `Tool` is a dataclass carrying its function in `fn`, so `dataclasses.replace` makes a copy with `fn` swapped for its `tracer.traced`-wrapped version — the originals stay clean, and nothing inside `shoplab` gets monkeypatched.

Two things here are deliberately *not* chapter 02's: the system prompt below is a fresh, shorter variant with no tool-call budget, and the ticket goes in as an indented JSON dump instead of the one-line rendering. The tracer could not care less — wrapping call sites works whatever the prompt says — but keep the difference in mind when you compare this run's outputs with chapter 02's.

In [ ]:
import json
from dataclasses import replace
from shoplab import world
from shoplab.loop import run_agent
from shoplab.tools import standard_tools

tracer.clear()
tools = {name: replace(t, fn=tracer.traced(name, kind="tool")(t.fn))
         for name, t in standard_tools().items()}
ticket = next(t for t in world.load_tickets()["train"]
              if t["ticket_id"] == "TKT-2205")
print(len(tools), "tools wrapped for", ticket["ticket_id"])

In [ ]:
FACTS = {k: ticket[k] for k in ("ticket_id", "order_id", "customer_id", "sku",
         "qty", "reason_text", "requested_action", "item_condition",
         "days_since_delivery", "evidence_photo")}
SYSTEM = ("You run the ops desk at Larkspur Outfitters. Triage the return "
          "ticket: look up the order and the customer, search the policies, "
          "compute any amount with calc, then report your decision by calling "
          "finish with decision, policy_id, and refund_usd. Do not move money "
          "yourself; finish is the only way to conclude.")

with tracer.span("triage", "agent", ticket=ticket["ticket_id"]) as root:
    result = run_agent("Triage this ticket:\n" + json.dumps(FACTS, indent=2),
                       tools, system=SYSTEM, max_steps=10)
    root.attributes["stop_reason"] = result.stop_reason
print(result.answer)

In [ ]:
tracer.print_tree()

> **What you should see:** a decision dict from the `finish` tool and, above it, the tree: `triage [agent]` at the margin with its `stop_reason` attribute filled in, and a handful of millisecond-scale `tool` spans indented beneath it — `get_order` and `get_customer` early, `search_policy` several times, `finish` last. On this ticket the trail typically ends at `pol-restocking` with about ten percent of the boot price held back, while the decision *label* is the least stable field — the frozen run above says `approve_refund` where chapter 02 said `partial_refund` over the same policy and amount. Resist reading that flip as a same-inputs experiment: this chapter also changed the prompt and the task rendering (flagged above), and underneath any prompt change temperature 0 only narrows sampling drift, it does not remove it. Chapter 04 holds one prompt still and grades the labels. A wandering run can still stop at `max_steps` — and this tree is then exactly how you find out where the step budget went.

Now notice what the tree hides. The tool spans sum to a few milliseconds, yet a live first run of this cell takes tens of seconds — nearly all the missing time is the model, invisible because you never wrapped the LLM call. The disk cache proves it from the other side: on a rerun the model turns replay from disk and the *same tree* collapses to a tenth of a second, tool spans unchanged. You could thread a span through `shoplab.llm.complete` to close the gap by hand; hold that thought instead, because the industrial half of this chapter closes it without you writing anything.

## The bridge is a POST

Phoenix's collector speaks OTLP, OpenTelemetry's wire format, and the payload is less exotic than the acronym: a `resource` (who is reporting — its `openinference.project.name` attribute is what Phoenix reads as the project to file everything under), a scope, and then the spans, each with ids as hex strings, times as *stringified nanoseconds*, and attributes as key/value records. `shoplab.trace.to_otlp` builds that envelope from a span list, minting one fresh 32-character trace id per export so the whole batch lands as a single trace.

One encoding wrinkle stands between JSON you can read and the collector: Phoenix accepts OTLP's binary protobuf encoding, not the JSON mapping, so `export_phoenix` converts the same tree mechanically (hex ids become raw bytes) and POSTs it to the server's `/v1/traces` route. Look at the payload first, then ship it.

In [ ]:
from shoplab.trace import to_otlp

payload = to_otlp(tracer.spans)
first = payload["resourceSpans"][0]["scopeSpans"][0]["spans"][0]
print("\n".join(json.dumps(first, indent=2).splitlines()[:9]))
print("resource:", json.dumps(payload["resourceSpans"][0]["resource"]))

In [ ]:
from shoplab.trace import export_phoenix

print("POST /v1/traces ->", export_phoenix(tracer))

> **What you should see:** in the payload, a 32-character `traceId`, 16-character span ids, and 19-digit nanosecond strings; from the export, `POST /v1/traces -> 200`. Now open the Phoenix link printed under the config cells. The project list shows `agentic-lab` — the name `to_otlp` stamped on the resource — and inside it sits the `triage` trace you just built: the same spans `print_tree` showed, same parent structure, now on a waterfall timeline. The dated `agentic-lab-` project next to it is the subject of the next section.

## The industrial version was already running

You have been running a second, better-funded tracer all along. Arize Phoenix is an open-source LLM tracing platform whose `phoenix.otel.register()` call collapses the raw OpenTelemetry setup into a single line, discovering and enabling OpenInference auto-instrumentation as it goes ([What is Arize Phoenix?](https://arize.com/docs/phoenix)). That is what `obs.enable_phoenix()` did at the top of this notebook: `register(auto_instrument=True)` found `openinference-instrumentation-litellm` in the environment and wrapped `litellm.completion` globally — so every model call this notebook has made, including the ones inside `run_agent` that your tracer rendered as gaps, is already a span in the dated project. Make one more, with nothing wrapped.

In [ ]:
from shoplab.llm import llm

print(llm("One sentence: what does an aircraft flight recorder capture?",
          max_tokens=60))

> **What you should see:** a one-sentence answer here — and in Phoenix, under the project named `agentic-lab-` plus today's date, a fresh `completion` span you never created, carrying the model string, prompt and completion token counts, latency, and the full message payloads. The agent run from earlier sits in the same project as a series of `completion` spans, roughly one per loop step. Token counts, latency, and model params on every call, recorded for free: that is what the packaging buys.

### What `obs.enable_phoenix()` actually promises

`obs.py` is a small robustness contract around that one `register()` call — worth reading once, because every chapter runs it:

| Contract | Mechanism |
|---|---|
| Dated project | everything logs under `agentic-lab-YYYY-MM-DD`, so different days never collide in the UI |
| Reuse before start | if the port is open and the body looks like Phoenix, the running server is adopted as-is |
| Never fight a stranger | a port held by something that is not Phoenix is skipped and the next port tried, four ports deep |
| Survives kernel restarts | a started server is a detached background process; rerunning the cell just finds it again |
| One-line routing | `phoenix.otel.register(project_name=..., auto_instrument=True, endpoint=...)` points OTLP at the chosen port |

## Two tracers, two jobs

Keep both. They are not competing implementations of one idea — they see different halves of the run:

| | Your `Tracer` | OpenInference autoinstrumentation |
|---|---|---|
| Sees | only what you wrap | every `litellm` call, wrapped for you |
| Records | your semantics: ticket ids, stop reasons, business steps | model, token counts, latency, full messages |
| Cost to add | one `with` line or decorator per step | zero, after `register(auto_instrument=True)` |
| Blind spot | the model — the gaps in your tree | your domain — which ticket, which policy won, why |

The rule that falls out: autoinstrument the plumbing, hand-span the business. No generic instrumentor will ever emit a `policy_lookup` span, tag a trace with `ticket=TKT-2205`, or record that a run stopped at `max_steps` — those are your semantics, and they are precisely the spans you will grep for when a refund goes wrong. Run both and the trace closes from both sides: LLM internals from the machinery, meaning from you.

## Recap

| Concept | One-liner |
|---|---|
| `Span` | name, kind, start/end, attributes, its own id, its parent's id — a tree node with a stopwatch. |
| `Tracer.span` | a contextmanager over a stack of open spans; the top of the stack is the next span's parent. |
| `@tracer.traced` | decorator form: one span per call, named after the function. |
| Call-site wrapping | agent span around `run_agent(...)`; `replace(t, fn=tracer.traced(...))` for tools; no internals patched. |
| `to_otlp` | resource, scope, spans: hex ids, nanosecond strings, key/value attributes; the resource attribute names the project. |
| `export_phoenix` | re-encodes the payload as protobuf and POSTs to `/v1/traces`; 200 means go look in the UI. |
| Autoinstrumentation | `obs.enable_phoenix()` runs `register(auto_instrument=True)`; every litellm call is traced for free. |
| Custom spans | your semantics — tickets, decisions, stop reasons — are invisible to generic instrumentors; wrap them yourself. |

## Exercises

1. Attach a business attribute no instrumentor could know: when opening the `triage` span, pass `queue_depth=` the number of train tickets not yet triaged, rerun the traced run, and re-export. Find the attribute in the Phoenix span details, then look at what the neighboring `completion` spans record and articulate why `queue_depth` could never appear there.
2. Trace and export a second ticket (any other train ticket works), then compare the two `triage` traces in Phoenix: tool-span counts, durations, and — in the dated project — how many `completion` spans each run added. Which ticket needed more model calls, and does the tool sequence alone explain why? (One extra agent run: well under a cent.)
3. Break the bridge on purpose: call `export_phoenix(tracer, endpoint=...)` pointed at a port nobody is listening on, then at the right port but a wrong path such as `/v1/nope`. One failure mode is an exception, the other a status code — establish which is which, and decide how `export_phoenix` should behave in each case if tracing must never take down the agent it observes.

**Next up:** chapter 04 puts numbers on quality — a deterministic rules engine computes the gold labels, and an eval harness grades the agent you just traced.